# E-DAXING — modern smartphones, both device representations

**The last major generality item.** Everything so far is Dresden: 2010-era CCD. Daxing is
2019-era smartphones with computational-photography pipelines — the regime where *Beyond PRNU*
predicts the low/mid-frequency signature matters most, and the regime the paper currently has to
concede.

### What this closes
| current limitation | after this |
|---|---|
| one dataset, 2010 CCD | two datasets, two sensor eras |
| "no a fortiori extension to modern devices" | **tested** |
| low/mid band shown on CCD only | tested where it should matter most |
| κ_model from 1 held-out body | **empirical κ from 8+ held-out same-model bodies** |

Daxing has **13 iPhone 6S bodies**, ≥482 images each — more same-model devices than any set used
so far. Five go into the crossed design; the rest give an empirical camera-model component.

### Two things that will go wrong if you skip D0
1. **Scene structure.** Daxing has few distinct scenes. Contiguous splits can put the same scene in
   the fingerprint and held-out splits, which inflates every correlation. D0 audits cross-split
   similarity and **halts** if it is too high.
2. **Filename convention.** The device-code regex below is a guess. D0 prints what it actually
   finds *before* anything else runs. **Inspect that output and fix `RX` if needed.**

Stages: **D0** discover/split/audit → **D1** fingerprints (PRNU *and* low/mid, plus empirical κ) →
**D2** VAE retention for both → **D3** 10 adapters → **D4** generation → **D5** measurement →
**D6** analysis. ~10 h GPU after D0/D1.

In [ ]:
# %% D-CFG
import os, sys, json, glob, re, math, time, gc, shutil, hashlib, subprocess, random, itertools
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import csv as _csv
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"

@dataclass
class CFG:
    STAGES: tuple = ("D0",)                 # inspect D0 first, then extend
    DAXING: str = "forensic_datasets/daxing"        # unpacked root
    OUT: str = "inv_channel/E_DAXING"
    MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CROP: int = 1024
    N_DEV: int = 5                          # devices in the crossed design
    N_HELDOUT_KAPPA: int = 6                # further same-model bodies, fingerprints only
    N_E1: int = 80; N_E2: int = 140; N_T: int = 50; N_H: int = 40; GUARD: int = 30
    SCENE_SIM_MAX: float = 0.95             # cross-split audit ceiling
    SEEDS: tuple = (0,1)                    # 5 devices x 2 seeds = 10 adapters
    RANK: int = 16
    TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    G_PER: int = 500; GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    BLK: int = 8; LO: int = 1; HI: int = 5  # low/mid band, identical to E-LOWFREQ
    N_PERM: int = 10000; ALPHA: float = 0.01
    GRAD_CKPT: bool = False
C=CFG(); MEAS=C.CROP
CFG_SHA=hashlib.sha256(json.dumps({k:v for k,v in asdict(C).items() if k!="STAGES"},
                                  sort_keys=True,default=str).encode()).hexdigest()
ROOT=os.path.join(DRIVE,C.OUT)
for d in ["","manifest","fingerprints","train_png","adapters","gens","csv","meta","figs"]:
    os.makedirs(os.path.join(ROOT,d),exist_ok=True)
CACHE="/content/dx_cache"; os.makedirs(CACHE,exist_ok=True)
def stage(n):
    if n in C.STAGES: return True
    print(f"[{n}] SKIPPED"); return False

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy.fft import dctn, idctn
from scipy import stats as sps
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
    print("[ENV] peft torchao dispatcher disabled")
except Exception: pass
Image.MAX_IMAGE_PIXELS=None
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise"); _MD=DEV
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _csv.DictReader(f)}
def ajson(o,p,**kw):
    t=p+".tmp"; json.dump(o,open(t,"w"),**kw); os.replace(t,p)
def DEVS(): return [f"P{i}" for i in range(C.N_DEV)]
def adir(t): return os.path.join(ROOT,"adapters",t)
def gdir(t): return os.path.join(ROOT,"gens",t)
def tdir(d): return os.path.join(ROOT,"train_png",d)
def all_arms(): return [(f"{d}_s{s}",d,s) for s in C.SEEDS for d in DEVS()]
def all_tags(): return [t for t,_,_ in all_arms()]
print(f"protocol SHA {CFG_SHA[:16]} | {len(all_arms())} adapters | device={DEV} dtype={DT}")

In [ ]:
# %% forensic core — PRNU and low/mid, both byte-identical to their home experiments
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        if min(W,H)<MEAS: HALT(f"{fp}: {W}x{H} < {MEAS}")
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):                       # PRNU: wavelet residual, db8 L4, sigma0=2
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
_MASK=None
def band_mask():
    global _MASK
    if _MASK is None:
        m=np.zeros((C.BLK,C.BLK),np.float32)
        for i in range(C.BLK):
            for j in range(C.BLK):
                if C.LO<=i+j<=C.HI and not (i==0 and j==0): m[i,j]=1.0
        _MASK=m; print(f"[band] keeping {int(m.sum())} of {C.BLK*C.BLK} coefficients per block")
    return _MASK
def lowmid(Y):                       # low/mid: block DCT band map
    n=MEAS//C.BLK
    B=Y.reshape(n,C.BLK,n,C.BLK).transpose(0,2,1,3)
    D=dctn(B,axes=(2,3),norm="ortho")*band_mask()[None,None]
    R=idctn(D,axes=(2,3),norm="ortho").transpose(0,2,1,3).reshape(MEAS,MEAS).astype(np.float32)
    R=R-R.mean(); sd=float(R.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate low/mid map")
    return (R/sd).astype(np.float32)
def zm(K):
    K=K-K.mean(axis=1,keepdims=True); return K-K.mean(axis=0,keepdims=True)
def wdft(K):
    F=np.fft.fft2(K.astype(np.float32)); mag=(np.abs(F)/K.shape[0]).astype(np.float32)
    sig=max(float(np.median(mag))/0.6745,1e-12); s2=np.float32(sig**2); vm=None
    for w in (3,5,7,9):
        k=np.ones((w,w),np.float32)/(w*w); mu=_c2(mag,k)
        v=np.maximum(_c2(mag*mag,k)-mu*mu,0.0); vm=v if vm is None else np.minimum(vm,v)
    return np.real(np.fft.ifft2(F*(s2/(vm+s2)).astype(np.float32))).astype(np.float32)
def estK(files):                     # PRNU MLE
    num=den=None
    for fp in files:
        Y=load_lum(fp); W=wres(Y)
        num=W*Y if num is None else num+W*Y; den=Y*Y if den is None else den+Y*Y
    return wdft(zm((num/np.maximum(den,1e-6)).astype(np.float32)))
def estL(files):                     # low/mid signature
    acc=None
    for fp in files:
        m=lowmid(load_lum(fp)); acc=m if acc is None else acc+m
    return wdft(zm((acc/len(files)).astype(np.float32)))
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
def thumb(fp,n=64):
    with Image.open(fp) as im:
        return np.asarray(im.convert("L").resize((n,n),Image.BILINEAR),np.float32).ravel()

In [ ]:
# %% D0 — INSPECT FIRST. Discovery, splits, and the cross-split scene audit.
#     The regex is a GUESS. Read the printout and fix RX before running anything else.
RX = re.compile(r"^(?P<code>[A-Za-z0-9]{4,6})[_\-]?(?P<shot>\d+)\.(jpe?g|JPE?G)$")
MODEL_OF = lambda code: code[:-1]        # last char = body index; ALSO A GUESS

def d0():
    base=os.path.join(DRIVE,C.DAXING)
    if not os.path.isdir(base): HALT(f"Daxing root not found: {base}")
    files=glob.glob(os.path.join(base,"**","*.*"),recursive=True)
    print(f"[D0] {len(files)} files under {base}")
    dv={}; unmatched=[]
    for fp in files:
        m=RX.match(os.path.basename(fp))
        if not m: unmatched.append(os.path.basename(fp)); continue
        dv.setdefault(m["code"],[]).append(fp)
    print(f"[D0] matched {sum(len(v) for v in dv.values())} files into {len(dv)} device codes")
    if unmatched:
        print(f"[D0] {len(unmatched)} UNMATCHED, first 10: {unmatched[:10]}")
    print("\n[D0] device codes found (code: n images) — CHECK THIS BEFORE PROCEEDING")
    for k,v in sorted(dv.items(), key=lambda kv:-len(kv[1]))[:30]:
        print(f"     {k:10s} model={MODEL_OF(k):8s} n={len(v)}")
    by_model={}
    for k,v in dv.items(): by_model.setdefault(MODEL_OF(k),[]).append((k,len(v)))
    need=C.N_E1+C.N_E2+C.N_T+C.N_H+3*C.GUARD
    print(f"\n[D0] same-model groups (need {need} images per body):")
    best=None
    for mo,bodies in sorted(by_model.items(), key=lambda kv:-len(kv[1])):
        ok=[b for b in bodies if b[1]>=need]
        print(f"     {mo:10s} {len(bodies)} bodies, {len(ok)} with >= {need} images")
        if len(ok)>=C.N_DEV and (best is None or len(ok)>len(best[1])): best=(mo,ok)
    if best is None: HALT("no camera model has enough usable bodies — check RX/MODEL_OF")
    model, usable = best
    usable=sorted(usable, key=lambda b:-b[1])
    print(f"\n[D0] SELECTED model {model}: {len(usable)} usable bodies")

    picks={}; kappa_bodies=[]
    for i,(code,n) in enumerate(usable):
        f=sorted(dv[code])
        cnt=Counter()
        for fp in f[:60]:
            try:
                with Image.open(fp) as im: cnt[im.size]+=1
            except Exception: pass
        size,_=cnt.most_common(1)[0]
        kept=[]
        for fp in f:
            try:
                with Image.open(fp) as im:
                    if im.size==size: kept.append(fp)
            except Exception: pass
        if min(size)<MEAS: print(f"     skip {code}: {size} < {MEAS}"); continue
        if i<C.N_DEV: picks[f"P{i}"]={"code":code,"size":list(size),"files":kept}
        elif len(kappa_bodies)<C.N_HELDOUT_KAPPA: kappa_bodies.append((code,kept))
        print(f"     {code}: {len(kept)} uniform @ {size}"
              f"{'  -> P%d'%i if i<C.N_DEV else ('  -> kappa' if len(kappa_bodies)<=C.N_HELDOUT_KAPPA else '')}")
    if len(picks)<C.N_DEV: HALT(f"only {len(picks)} usable bodies after the resolution filter")

    # contiguous splits with LARGE guards (Daxing is scene-ordered), then AUDIT
    hdr=["dev","code","split","idx","path"]; cp=os.path.join(ROOT,"manifest","manifest.csv")
    if os.path.exists(cp): os.remove(cp)
    man={"config_sha":CFG_SHA,"model":model,"picks":{},"kappa_bodies":[b[0] for b in kappa_bodies],
         "meas":MEAS,"guard":C.GUARD}
    worst=0.0
    for d,info in picks.items():
        k=info["files"]; g=C.GUARD; i=0; sp={}
        for nm,n in (("E1",C.N_E1),("E2",C.N_E2),("T",C.N_T),("H",C.N_H)):
            sp[nm]=k[i:i+n]; i+=n+g
        # scene audit: max cosine similarity between split thumbnails
        th={nm:np.stack([thumb(x) for x in v[::max(1,len(v)//20)][:20]]) for nm,v in sp.items()}
        for a,b in itertools.combinations(th,2):
            X=th[a]/np.linalg.norm(th[a],axis=1,keepdims=True)
            Y=th[b]/np.linalg.norm(th[b],axis=1,keepdims=True)
            s=float((X@Y.T).max()); worst=max(worst,s)
            if s>C.SCENE_SIM_MAX:
                print(f"     [!] {d} {a}-{b} thumbnail similarity {s:.3f} > {C.SCENE_SIM_MAX}")
        for nm,v in sp.items():
            for j,fp in enumerate(v): append_row(cp,hdr,[d,info["code"],nm,j,fp])
        man["picks"][d]={"code":info["code"],"size":info["size"],
                         "n":{k_:len(v) for k_,v in sp.items()}}
        print(f"     {d} ({info['code']}): splits "
              f"{ {k_:len(v) for k_,v in sp.items()} }")
    for code,kept in kappa_bodies:
        for j,fp in enumerate(kept[:C.N_E2]): append_row(cp,hdr,[f"K_{code}",code,"E2",j,fp])
    ajson(man,os.path.join(ROOT,"manifest","manifest.json"),indent=2)
    print(f"\n[D0] worst cross-split thumbnail similarity = {worst:.3f} "
          f"(ceiling {C.SCENE_SIM_MAX})")
    if worst>C.SCENE_SIM_MAX:
        HALT("cross-split scene leakage: raise GUARD, or split by scene label if available")
    print(f"[D0] PASS — {len(picks)} devices + {len(kappa_bodies)} kappa bodies, model {model}")
if stage("D0"): d0()

In [ ]:
# %% D1 — fingerprints in BOTH representations + an EMPIRICAL camera-model component
def MANIF():
    M={}
    for r in _csv.DictReader(open(os.path.join(ROOT,"manifest","manifest.csv"))):
        M.setdefault((r["dev"],r["split"]),[]).append(r["path"])
    return M
def d1():
    M=MANIF(); man=json.load(open(os.path.join(ROOT,"manifest","manifest.json")))
    for d in DEVS():
        for which in ("E1","E2"):
            for tag,fn in (("K",estK),("L",estL)):
                p=os.path.join(ROOT,"fingerprints",f"{tag}_{d}_{which}.npy")
                if os.path.exists(p): continue
                t0=time.time(); np.save(p,fn(M[(d,which)]))
                print(f"[D1] {tag}_{d}_{which} from {len(M[(d,which)])} imgs ({time.time()-t0:.0f}s)")
    # EMPIRICAL kappa_model from the held-out bodies — what 13 same-model devices buy
    kb=[k for k in man["kappa_bodies"]]
    for tag,fn in (("K",estK),("L",estL)):
        acc=[]
        for code in kb:
            p=os.path.join(ROOT,"fingerprints",f"{tag}_hold_{code}.npy")
            if not os.path.exists(p):
                t0=time.time(); np.save(p,fn(M[(f"K_{code}","E2")]))
                print(f"[D1] {tag}_hold_{code} ({time.time()-t0:.0f}s)")
            acc.append(np.load(p))
        kap=np.mean(acc,axis=0).astype(np.float32)
        np.save(os.path.join(ROOT,"fingerprints",f"{tag}_kappa_empirical.npy"),kap)
        pair=[ncc(acc[i],acc[j]) for i,j in itertools.combinations(range(len(acc)),2)]
        print(f"\n[D1] {tag}: EMPIRICAL kappa_model from {len(acc)} held-out bodies")
        print(f"     pairwise NCC between held-out bodies: mean {np.mean(pair):+.4f} "
              f"sd {np.std(pair):.4f} max {np.max(pair):+.4f}")
    # split-half reliability + same-model NCC, both representations
    print(f"\n[D1] {'rep':6s} {'split-half NCC(E1,E2)':>22s} {'mean pairwise NCC across devices':>34s}")
    for tag in ("K","L"):
        sh=[ncc(np.load(os.path.join(ROOT,"fingerprints",f"{tag}_{d}_E1.npy")),
                np.load(os.path.join(ROOT,"fingerprints",f"{tag}_{d}_E2.npy"))) for d in DEVS()]
        F={d:np.load(os.path.join(ROOT,"fingerprints",f"{tag}_{d}_E2.npy")) for d in DEVS()}
        cross=[ncc(F[a],F[b]) for a,b in itertools.combinations(DEVS(),2)]
        print(f"     {tag:6s} {np.mean(sh):22.4f} {np.mean(cross):34.4f}")
        print(f"            (Dresden PRNU 0.362 / low-mid 0.0812)"
              if tag=="K" else "")
    # positive control on H, both representations
    for tag,mapf in (("K",wres),("L",lowmid)):
        p=os.path.join(ROOT,"csv",f"d1_positive_{tag}.csv")
        if os.path.exists(p): continue
        F={d:np.load(os.path.join(ROOT,"fingerprints",f"{tag}_{d}_E2.npy")) for d in DEVS()}
        for d in DEVS():
            for i,fp in enumerate(M[(d,"H")]):
                Y=load_lum(fp); W=mapf(Y)
                for k_ in DEVS():
                    append_row(p,["img_dev","F_dev","idx","rho"],[d,k_,i,f"{ncc(W,Y*F[k_]):.6e}"])
            print(f"[D1] positive control {tag} {d} done")
    print()
    for tag in ("K","L"):
        pc=pd.read_csv(os.path.join(ROOT,"csv",f"d1_positive_{tag}.csv"))
        same=pc[pc.img_dev==pc.F_dev]["rho"].to_numpy(); cross=pc[pc.img_dev!=pc.F_dev]["rho"].to_numpy()
        lab=np.r_[np.ones(len(same)),np.zeros(len(cross))]; sc=np.r_[same,cross]
        o=np.argsort(sc); rk=np.empty(len(sc)); rk[o]=np.arange(len(sc))
        auc=float((rk[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum()))
        per=[]
        for d in DEVS():
            g=pc[pc.img_dev==d]
            own=g[g.F_dev==d].sort_values("idx")["rho"].to_numpy()
            oth=np.mean([g[g.F_dev==o_].sort_values("idx")["rho"].to_numpy()
                         for o_ in DEVS() if o_!=d],axis=0)
            n=min(len(own),len(oth)); per.append(float((own[:n]-oth[:n]).mean()))
        per=np.array(per); R=float(per.mean()); se=float(per.std(ddof=1)/np.sqrt(len(per)))
        print(f"[D1] {tag}: same-vs-cross AUC {auc:.4f} | R_real {R:.5e} "
              f"lower99 {R-sps.norm.ppf(0.99)*se:.5e}")
        ajson({"auc":auc,"R_real":R,"se":se,
               "R_real_lower99":float(R-sps.norm.ppf(0.99)*se),"per_device":per.tolist()},
              os.path.join(ROOT,"fingerprints",f"d1_gates_{tag}.json"),indent=2)
        if tag=="K" and auc<0.99: HALT(f"PRNU gate: AUC {auc:.4f} < 0.99 on modern smartphones")
        if tag=="L" and auc<0.75: print("     [!] low/mid gate failed on this dataset — report it")
    print("[D1] PASS")
if stage("D1"): d1()

## D2–D6

`D2` (VAE retention, both representations), `D3` (10 adapters), `D4` (generation), `D5`
(measurement in both representations) and `D6` (symmetric-statistic analysis, additive
decomposition, bounds) are **structurally identical to `E_TRACK_C` stages C1–C4 and `E_LOWFREQ`
L1–L3**, with `K_*`/`L_*` in place of the Dresden fingerprints. Copy those cells across once D0
and D1 have passed and the manifest is confirmed — the shapes and paths already match.

Two things to carry over deliberately:
- **D6 must use the symmetric statistic**, not per-arm `max(U)`. The low/mid band showed why: its
  per-arm bound was 20.6% and additive-contaminated, against 9.7% from the symmetric form.
- **Report the additive/interaction split** for both representations. On Dresden the additive term
  was ~0 for PRNU and 9.7× the interaction for low/mid; whether that pattern holds on modern
  smartphones is a real question, not a formality.

## Running D0/D1

| stage | wall | check |
|---|---|---|
| **D0** | ~10 min | **the regex printout** — fix `RX`/`MODEL_OF` if the device codes look wrong; then the cross-split thumbnail audit |
| **D1** | ~2 h | PRNU AUC ≥ 0.99 (gate); low/mid AUC vs Dresden's 0.8433; split-half reliability vs 0.362 / 0.0812; **empirical κ_model from 6 held-out bodies** |

**The empirical κ is what 13 same-model bodies buy and Dresden could not give.** On Dresden, κ was
a single NCC between two devices (0.0073); here it is a mean over 6 independent bodies with a
pairwise spread, so the camera-model component becomes an estimated quantity rather than a point
comparison.

**If the low/mid gate fails on smartphones**, that is itself worth reporting: the band that looked
more robust on CCD does not survive a computational-photography pipeline. Do not tune `HI` to
rescue it.